# SEC 13F Holdings Data Analysis

Exploratory analysis of institutional fund holdings from SEC 13F filings.

**Database:** Social_13F | **Table:** holdings_filtered_new

In [7]:
pip install seaborn

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import sys
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=UserWarning)

# Load environment variables
env_paths = [
    Path(r'C:\Users\potda\Daniel\BGU\Year_D\Final_Project\Social-Network-Stock-Market\.env'),
    Path.cwd() / '.env',
    Path.cwd().parent / '.env',
    Path.cwd().parent.parent / '.env',
]
for env_path in env_paths:
    if env_path.exists():
        load_dotenv(env_path)
        print(f"Loaded .env from: {env_path}")
        break

# Add project root to path
notebook_path = Path.cwd()
project_root = notebook_path.parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from ETL.data_handlers.db_data_handler.postgres_handler import PostgresHandler

print(f"Project root: {project_root}")
print(f"DB_HOST: {os.getenv('DB_HOST', 'Not set')}")
print(f"DB_NAME: {os.getenv('DB_NAME', 'Not set')}")

Loaded .env from: c:\Users\potda\Daniel\BGU\Year_D\סמסטר ז\Final_Project\Social-Network-Stock-Market\.env
Project root: c:\Users\potda\Daniel\BGU\Year_D\סמסטר ז\Final_Project\Social-Network-Stock-Market
DB_HOST: 127.0.0.1
DB_NAME: Social_13F


In [9]:
# Style Configuration
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)
PALETTE = sns.color_palette("mako", 12)
ACCENT = sns.color_palette("flare", 12)

IMAGES_DIR = Path("images")
IMAGES_DIR.mkdir(exist_ok=True)


def save_fig(fig, name: str, dpi=300):
    """Save figure to images directory as high-res PNG."""
    path = IMAGES_DIR / f"{name}.png"
    fig.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    print(f"Saved: {path}")


def format_quarter_label(year, quarter):
    return f"{int(year)} Q{int(quarter)}"


print(f"Images will be saved to: {IMAGES_DIR.resolve()}")

Images will be saved to: C:\Users\potda\Daniel\BGU\Year_D\סמסטר ז\Final_Project\Social-Network-Stock-Market\Notebooks\data_analysis\images


In [11]:
# Database Connection & Data Loading

handler = PostgresHandler()
if not handler.connect():
    raise ConnectionError("Failed to connect to database")
print(f"Connected to database: {handler.database}")

# Query 1: Core holdings aggregation (for counting unique entities, portfolio sizes, etc.)
query_core = """
SELECT 
    hf.cik, hf.cusip, hf.year, hf.quarter,
    SUM(hf.sshprnamt) AS total_shares,
    COUNT(*) AS num_holdings
FROM holdings_filtered_new hf
WHERE hf.year IS NOT NULL AND hf.quarter IS NOT NULL
GROUP BY hf.cik, hf.cusip, hf.year, hf.quarter
"""

# Query 2: Holdings with price data (for value-based analysis)
query_valued = """
SELECT 
    hf.cik, hf.cusip, hf.year, hf.quarter,
    SUM(hf.sshprnamt * tp.price) AS holding_value,
    SUM(hf.sshprnamt) AS total_shares
FROM holdings_filtered_new hf
INNER JOIN ticker_to_cusip ttc ON hf.cusip = ttc.cusip
INNER JOIN ticker_prices tp ON ttc.ticker = tp.ticker 
    AND hf.period_start = CAST(tp.period_start AS DATE)
WHERE hf.sshprnamt IS NOT NULL AND hf.sshprnamt > 0
    AND tp.price IS NOT NULL AND tp.price > 0
GROUP BY hf.cik, hf.cusip, hf.year, hf.quarter
"""

# Query 3: CUSIP to issuer name mapping
query_names = """
SELECT DISTINCT cusip, nameofissuer 
FROM holdings_filtered_new 
WHERE nameofissuer IS NOT NULL
"""

print("Loading core holdings data...")
df_core = pd.read_sql_query(query_core, handler.connection)
print(f"  Core data: {len(df_core):,} rows")

print("Loading valued holdings data...")
df_valued = pd.read_sql_query(query_valued, handler.connection)
print(f"  Valued data: {len(df_valued):,} rows")

print("Loading issuer names...")
df_names = pd.read_sql_query(query_names, handler.connection)
print(f"  Names: {len(df_names):,} unique CUSIPs")

handler.disconnect()
print("\nDatabase connection closed.")

2026-03-22 23:23:30 - ETL_Pipeline - INFO - Connected to PostgreSQL: postgres@127.0.0.1:5432/Social_13F


Connected to database: Social_13F
Loading core holdings data...


KeyboardInterrupt: 

In [ ]:
# Data Preparation

for df in [df_core, df_valued]:
    df['quarter_label'] = df.apply(lambda r: format_quarter_label(r['year'], r['quarter']), axis=1)
    df['sort_key'] = df['year'] * 10 + df['quarter']

# CUSIP name lookup
cusip_to_name = df_names.drop_duplicates('cusip').set_index('cusip')['nameofissuer'].to_dict()

# All unique quarters sorted
all_quarters = (df_core[['year', 'quarter', 'quarter_label', 'sort_key']]
                .drop_duplicates()
                .sort_values('sort_key')
                .reset_index(drop=True))

print(f"Data spans {len(all_quarters)} quarters")
print(f"  From: {all_quarters.iloc[0]['quarter_label']}")
print(f"  To:   {all_quarters.iloc[-1]['quarter_label']}")
print(f"\nUnique funds (CIKs): {df_core['cik'].nunique():,}")
print(f"Unique stocks (CUSIPs): {df_core['cusip'].nunique():,}")
print(f"Total rows (core): {len(df_core):,}")
print(f"Total rows (valued): {len(df_valued):,}")

---
## 1. Unique CUSIPs (Stocks) per Year and Quarter
How many distinct stocks are held by institutional funds each period?

In [ ]:
# Unique CUSIPs per quarter
cusips_per_q = (df_core.groupby(['year', 'quarter', 'quarter_label', 'sort_key'])['cusip']
                .nunique()
                .reset_index(name='unique_cusips')
                .sort_values('sort_key'))

# Unique CUSIPs per year
cusips_per_y = (df_core.groupby('year')['cusip']
                .nunique()
                .reset_index(name='unique_cusips'))

fig, axes = plt.subplots(1, 2, figsize=(18, 6), facecolor='white')

# Left: quarterly line
ax = axes[0]
ax.plot(range(len(cusips_per_q)), cusips_per_q['unique_cusips'],
        marker='o', color=PALETTE[3], linewidth=2.5, markersize=5, zorder=3)
ax.fill_between(range(len(cusips_per_q)), cusips_per_q['unique_cusips'],
                alpha=0.15, color=PALETTE[3])
step = max(1, len(cusips_per_q) // 10)
ax.set_xticks(range(0, len(cusips_per_q), step))
ax.set_xticklabels(cusips_per_q['quarter_label'].iloc[::step], rotation=45, ha='right')
ax.set_title('Unique CUSIPs per Quarter', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Unique CUSIPs')
ax.set_xlabel('Quarter')
ax.grid(alpha=0.3)

# Right: yearly bar
ax = axes[1]
bars = ax.bar(cusips_per_y['year'].astype(str), cusips_per_y['unique_cusips'],
              color=PALETTE[5], edgecolor='white', linewidth=1.2)
ax.bar_label(bars, fontsize=9, padding=3, fmt='%d')
ax.set_title('Unique CUSIPs per Year', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Unique CUSIPs')
ax.set_xlabel('Year')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
ax.grid(alpha=0.3)

plt.suptitle('Stock Universe Coverage Over Time', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, '01_unique_cusips_per_period')
plt.show()

---
## 2. Unique CIKs (Funds) per Year and Quarter
How many distinct institutional funds file 13F reports each period?

In [ ]:
# Unique CIKs per quarter
ciks_per_q = (df_core.groupby(['year', 'quarter', 'quarter_label', 'sort_key'])['cik']
              .nunique()
              .reset_index(name='unique_ciks')
              .sort_values('sort_key'))

# Unique CIKs per year
ciks_per_y = (df_core.groupby('year')['cik']
              .nunique()
              .reset_index(name='unique_ciks'))

fig, axes = plt.subplots(1, 2, figsize=(18, 6), facecolor='white')

# Left: quarterly line
ax = axes[0]
ax.plot(range(len(ciks_per_q)), ciks_per_q['unique_ciks'],
        marker='s', color=PALETTE[6], linewidth=2.5, markersize=5, zorder=3)
ax.fill_between(range(len(ciks_per_q)), ciks_per_q['unique_ciks'],
                alpha=0.15, color=PALETTE[6])
step = max(1, len(ciks_per_q) // 10)
ax.set_xticks(range(0, len(ciks_per_q), step))
ax.set_xticklabels(ciks_per_q['quarter_label'].iloc[::step], rotation=45, ha='right')
ax.set_title('Unique CIKs per Quarter', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Unique CIKs')
ax.set_xlabel('Quarter')
ax.grid(alpha=0.3)

# Right: yearly bar
ax = axes[1]
bars = ax.bar(ciks_per_y['year'].astype(str), ciks_per_y['unique_ciks'],
              color=PALETTE[8], edgecolor='white', linewidth=1.2)
ax.bar_label(bars, fontsize=9, padding=3, fmt='%d')
ax.set_title('Unique CIKs per Year', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Unique CIKs')
ax.set_xlabel('Year')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
ax.grid(alpha=0.3)

plt.suptitle('Fund Participation Over Time', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, '02_unique_ciks_per_period')
plt.show()

---
## 3. Top 10 Highest Value CIKs (Funds)
Which funds have the largest total portfolio value across all time?

In [ ]:
# Top 10 CIKs by total portfolio value (sum across all quarters)
top_ciks = (df_valued.groupby('cik')['holding_value']
            .sum()
            .nlargest(10)
            .reset_index()
            .sort_values('holding_value', ascending=True))

fig, ax = plt.subplots(figsize=(12, 7), facecolor='white')

colors = sns.color_palette("mako", len(top_ciks))
bars = ax.barh(top_ciks['cik'], top_ciks['holding_value'] / 1e9,
               color=colors, edgecolor='white', linewidth=1.2)

ax.bar_label(bars, fmt='$%.1fB', fontsize=9, padding=5)
ax.set_xlabel('Total Portfolio Value (Billions USD)', fontsize=12)
ax.set_ylabel('CIK (Fund Identifier)', fontsize=12)
ax.set_title('Top 10 Funds by Total Portfolio Value (All Time)',
             fontsize=14, fontweight='bold')
ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
save_fig(fig, '03_top10_ciks_by_value')
plt.show()

---
## 4. Top Stocks by Average Quarterly Investment
Which stocks receive the highest investment on average per quarter?

In [ ]:
# Total value per CUSIP per quarter, then average across quarters
cusip_quarterly = (df_valued.groupby(['cusip', 'year', 'quarter'])['holding_value']
                   .sum()
                   .reset_index())

cusip_avg = (cusip_quarterly.groupby('cusip')['holding_value']
             .mean()
             .nlargest(15)
             .reset_index()
             .sort_values('holding_value', ascending=True))

# Add issuer names as labels
cusip_avg['label'] = cusip_avg['cusip'].map(
    lambda c: f"{cusip_to_name.get(c, 'Unknown')[:25]} ({c})")

fig, ax = plt.subplots(figsize=(13, 8), facecolor='white')

colors = sns.color_palette("flare", len(cusip_avg))
bars = ax.barh(cusip_avg['label'], cusip_avg['holding_value'] / 1e9,
               color=colors, edgecolor='white', linewidth=1.2)

ax.bar_label(bars, fmt='$%.1fB', fontsize=9, padding=5)
ax.set_xlabel('Average Quarterly Investment (Billions USD)', fontsize=12)
ax.set_title('Top 15 Stocks by Average Quarterly Investment',
             fontsize=14, fontweight='bold')
ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
save_fig(fig, '04_top_cusips_avg_quarterly_investment')
plt.show()

---
## 5. Total Money Invested per Quarter
What is the total dollar value of all institutional holdings each quarter?

In [ ]:
# Total investment value per quarter
money_per_q = (df_valued.groupby(['year', 'quarter'])['holding_value']
               .sum()
               .reset_index()
               .sort_values(['year', 'quarter']))
money_per_q['quarter_label'] = money_per_q.apply(
    lambda r: format_quarter_label(r['year'], r['quarter']), axis=1)

fig, ax = plt.subplots(figsize=(16, 7), facecolor='white')

ax.fill_between(range(len(money_per_q)), money_per_q['holding_value'] / 1e12,
                alpha=0.2, color=PALETTE[4])
ax.plot(range(len(money_per_q)), money_per_q['holding_value'] / 1e12,
        marker='o', linewidth=2.5, color=PALETTE[4], markersize=6, zorder=3)

step = max(1, len(money_per_q) // 12)
ax.set_xticks(range(0, len(money_per_q), step))
ax.set_xticklabels(money_per_q['quarter_label'].iloc[::step], rotation=45, ha='right')
ax.set_ylabel('Total Investment (Trillions USD)', fontsize=12)
ax.set_xlabel('Quarter', fontsize=12)
ax.set_title('Total Money Invested per Quarter', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.1fT'))
ax.grid(alpha=0.3)

plt.tight_layout()
save_fig(fig, '05_total_money_per_quarter')
plt.show()

---
## 6. Distribution of Portfolio Sizes (Stocks per Fund)
How diversified are institutional portfolios? How many stocks does a typical fund hold?

In [ ]:
# Number of unique stocks each fund holds per quarter
portfolio_sizes = (df_core.groupby(['cik', 'year', 'quarter'])['cusip']
                   .nunique()
                   .reset_index(name='num_stocks'))

fig, axes = plt.subplots(1, 2, figsize=(18, 6), facecolor='white')

# Left: histogram
ax = axes[0]
ax.hist(portfolio_sizes['num_stocks'], bins=50, color=PALETTE[6],
        edgecolor='white', alpha=0.85)
median_val = portfolio_sizes['num_stocks'].median()
mean_val = portfolio_sizes['num_stocks'].mean()
ax.axvline(median_val, color='red', linestyle='--', linewidth=2,
           label=f'Median: {median_val:.0f} stocks')
ax.axvline(mean_val, color='orange', linestyle='--', linewidth=2,
           label=f'Mean: {mean_val:.0f} stocks')
ax.set_xlabel('Number of Stocks Held', fontsize=12)
ax.set_ylabel('Frequency (fund-quarter observations)', fontsize=12)
ax.set_title('Distribution of Portfolio Sizes', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# Right: box plot by year
ax = axes[1]
sns.boxplot(data=portfolio_sizes, x='year', y='num_stocks', ax=ax,
            palette='mako', showfliers=False, linewidth=1.2)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Stocks per Fund', fontsize=12)
ax.set_title('Portfolio Size Distribution by Year', fontsize=14, fontweight='bold')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
ax.grid(alpha=0.3)

plt.suptitle('Fund Diversification Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, '06_portfolio_size_distribution')
plt.show()

---
## 7. Market Concentration: Top 10 Stocks' Share of Total Investment
What percentage of total investment is concentrated in the top 10 most-held stocks each quarter?

In [ ]:
# Per quarter: what % of total investment goes to top 10 CUSIPs
cusip_q_totals = (df_valued.groupby(['year', 'quarter', 'cusip'])['holding_value']
                  .sum()
                  .reset_index())


def top_n_concentration(group, n=10):
    total = group['holding_value'].sum()
    top_n = group.nlargest(n, 'holding_value')['holding_value'].sum()
    return top_n / total * 100 if total > 0 else 0


concentration = (cusip_q_totals.groupby(['year', 'quarter'])
                 .apply(top_n_concentration)
                 .reset_index(name='top10_pct')
                 .sort_values(['year', 'quarter']))
concentration['quarter_label'] = concentration.apply(
    lambda r: format_quarter_label(r['year'], r['quarter']), axis=1)

fig, ax = plt.subplots(figsize=(16, 6), facecolor='white')

ax.bar(range(len(concentration)), concentration['top10_pct'],
       color=PALETTE[7], edgecolor='white', alpha=0.85, linewidth=0.8)
avg_conc = concentration['top10_pct'].mean()
ax.axhline(avg_conc, color='red', linestyle='--', linewidth=2,
           label=f'Mean: {avg_conc:.1f}%')

step = max(1, len(concentration) // 12)
ax.set_xticks(range(0, len(concentration), step))
ax.set_xticklabels(concentration['quarter_label'].iloc[::step], rotation=45, ha='right')
ax.set_ylabel('% of Total Investment in Top 10 Stocks', fontsize=12)
ax.set_xlabel('Quarter', fontsize=12)
ax.set_title('Market Concentration: Top 10 Stocks\' Share of Total Investment',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
save_fig(fig, '07_concentration_top10')
plt.show()

---
## 8. New Entrants vs Exits per Quarter
How many funds and stocks enter or leave the market each quarter?

In [ ]:
# Track entities appearing/disappearing each quarter
def compute_entries_exits(df, entity_col):
    entities_per_q = (df.groupby(['year', 'quarter'])[entity_col]
                      .apply(set)
                      .reset_index(name='entities')
                      .sort_values(['year', 'quarter']))

    results = []
    prev_set = set()
    for _, row in entities_per_q.iterrows():
        current = row['entities']
        new_count = len(current - prev_set) if prev_set else 0
        exit_count = len(prev_set - current) if prev_set else 0
        results.append({
            'year': row['year'], 'quarter': row['quarter'],
            'new': new_count, 'exited': -exit_count
        })
        prev_set = current
    return pd.DataFrame(results[1:])  # skip first quarter


cik_changes = compute_entries_exits(df_core, 'cik')
cusip_changes = compute_entries_exits(df_core, 'cusip')

fig, axes = plt.subplots(2, 1, figsize=(16, 10), facecolor='white')

for ax, data, title in [(axes[0], cik_changes, 'Funds (CIKs)'),
                         (axes[1], cusip_changes, 'Stocks (CUSIPs)')]:
    data = data.copy()
    data['qlabel'] = data.apply(
        lambda r: format_quarter_label(r['year'], r['quarter']), axis=1)
    x = range(len(data))
    ax.bar(x, data['new'], color=PALETTE[3], label='New Entrants',
           alpha=0.85, edgecolor='white')
    ax.bar(x, data['exited'], color=ACCENT[3], label='Exits',
           alpha=0.85, edgecolor='white')
    ax.axhline(0, color='black', linewidth=0.8)
    step = max(1, len(data) // 10)
    ax.set_xticks(range(0, len(data), step))
    ax.set_xticklabels(data['qlabel'].iloc[::step], rotation=45, ha='right')
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title(f'{title}: New Entrants vs Exits per Quarter',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

plt.tight_layout()
save_fig(fig, '08_entrants_vs_exits')
plt.show()

---
## 9. Bipartite Network Density Over Time
How dense is the fund-stock network each quarter? (actual edges / possible edges)

In [ ]:
# Network density: edges / (num_ciks * num_cusips) per quarter
network_stats = (df_core.groupby(['year', 'quarter'])
                 .agg(
                     num_edges=('cusip', 'size'),
                     num_ciks=('cik', 'nunique'),
                     num_cusips=('cusip', 'nunique')
                 )
                 .reset_index()
                 .sort_values(['year', 'quarter']))

network_stats['possible_edges'] = network_stats['num_ciks'] * network_stats['num_cusips']
network_stats['density'] = network_stats['num_edges'] / network_stats['possible_edges']
network_stats['quarter_label'] = network_stats.apply(
    lambda r: format_quarter_label(r['year'], r['quarter']), axis=1)

fig, axes = plt.subplots(1, 2, figsize=(18, 6), facecolor='white')

# Left: density over time
ax = axes[0]
ax.plot(range(len(network_stats)), network_stats['density'] * 100,
        marker='s', linewidth=2.5, color=PALETTE[8], markersize=5, zorder=3)
ax.fill_between(range(len(network_stats)), network_stats['density'] * 100,
                alpha=0.15, color=PALETTE[8])
step = max(1, len(network_stats) // 10)
ax.set_xticks(range(0, len(network_stats), step))
ax.set_xticklabels(network_stats['quarter_label'].iloc[::step], rotation=45, ha='right')
ax.set_ylabel('Network Density (%)', fontsize=12)
ax.set_xlabel('Quarter', fontsize=12)
ax.set_title('Bipartite Network Density Over Time', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)

# Right: number of edges over time
ax = axes[1]
ax.bar(range(len(network_stats)), network_stats['num_edges'] / 1000,
       color=PALETTE[5], edgecolor='white', alpha=0.85)
ax.set_xticks(range(0, len(network_stats), step))
ax.set_xticklabels(network_stats['quarter_label'].iloc[::step], rotation=45, ha='right')
ax.set_ylabel('Number of Edges (thousands)', fontsize=12)
ax.set_xlabel('Quarter', fontsize=12)
ax.set_title('Total Fund-Stock Connections per Quarter', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)

plt.suptitle('Network Structure Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, '09_network_density')
plt.show()

---
## 10. Fund Portfolio Overlap Heatmap
How similar are the portfolios of the largest funds? (Jaccard similarity of their stock holdings in the latest quarter)

In [ ]:
# Use the latest quarter for overlap analysis
latest_q = all_quarters.iloc[-1]
latest_data = df_core[(df_core['year'] == latest_q['year']) &
                       (df_core['quarter'] == latest_q['quarter'])]

# Pick top 20 funds by number of holdings
top_funds = (latest_data.groupby('cik')['cusip'].nunique()
             .nlargest(20).index.tolist())
fund_data = latest_data[latest_data['cik'].isin(top_funds)]

# Build fund -> set of CUSIPs
fund_stocks = fund_data.groupby('cik')['cusip'].apply(set).to_dict()

# Compute Jaccard similarity matrix
funds = list(fund_stocks.keys())
n = len(funds)
jaccard = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        intersection = len(fund_stocks[funds[i]] & fund_stocks[funds[j]])
        union = len(fund_stocks[funds[i]] | fund_stocks[funds[j]])
        jaccard[i, j] = intersection / union if union > 0 else 0

jaccard_df = pd.DataFrame(jaccard, index=funds, columns=funds)

fig, ax = plt.subplots(figsize=(14, 12), facecolor='white')
sns.heatmap(jaccard_df, annot=True, fmt='.2f', cmap='mako', ax=ax,
            linewidths=0.5, square=True, vmin=0, vmax=1,
            cbar_kws={'label': 'Jaccard Similarity', 'shrink': 0.8},
            annot_kws={'fontsize': 7})
ax.set_title(f'Fund Portfolio Overlap - Top 20 Funds ({latest_q["quarter_label"]})',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('CIK', fontsize=12)
ax.set_ylabel('CIK', fontsize=12)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
plt.setp(ax.get_yticklabels(), rotation=0, fontsize=8)

plt.tight_layout()
save_fig(fig, '10_fund_overlap_heatmap')
plt.show()

---
## Summary Statistics

In [ ]:
# Final summary
print("=" * 60)
print("  SEC 13F Holdings - Data Summary")
print("=" * 60)
print(f"  Quarters analyzed:        {len(all_quarters)}")
print(f"  Date range:               {all_quarters.iloc[0]['quarter_label']} - {all_quarters.iloc[-1]['quarter_label']}")
print(f"  Unique funds (CIKs):      {df_core['cik'].nunique():,}")
print(f"  Unique stocks (CUSIPs):   {df_core['cusip'].nunique():,}")
print(f"  Total holdings (core):    {len(df_core):,}")
print(f"  Total holdings (valued):  {len(df_valued):,}")
if not df_valued.empty:
    total_val = df_valued['holding_value'].sum()
    print(f"  Total investment (all time): ${total_val/1e12:,.2f} Trillion")
print("=" * 60)
print(f"\n  All 10 figures saved to: {IMAGES_DIR.resolve()}")